# Met Office — Land Observations (minimal)

Minimal pipeline: fetch land observations for a geohash and convert to a tabular DataFrame.

Environment variables (set in `.env`):
- `MET_OFFICE_API_KEY`: your Met Office API key (required)
- `MET_OFFICE_GEOHASH`: location geohash (required)
- `MET_OFFICE_LAND_API_BASE` (optional): API base, defaults to https://data.hub.api.metoffice.gov.uk

Run cells in order: 1) this explanation, 2) Setup, 3) Fetch, 4) Parse (produces `df`).

In [6]:
# --- Setup: imports and environment ---
import os
from dotenv import load_dotenv

load_dotenv()

MET_OFFICE_API_KEY = os.getenv('MET_OFFICE_API_KEY')
MET_OFFICE_GEOHASH = os.getenv('MET_OFFICE_GEOHASH')
MET_OFFICE_LAND_API_BASE = os.getenv('MET_OFFICE_LAND_API_BASE', 'https://data.hub.api.metoffice.gov.uk')

missing = []
if not MET_OFFICE_API_KEY:
    missing.append('MET_OFFICE_API_KEY')
if not MET_OFFICE_GEOHASH:
    missing.append('MET_OFFICE_GEOHASH')
if missing:
    raise AssertionError(f"Missing environment variables: {', '.join(missing)}. Set them in .env and re-run this cell.")

# show concise info (no full secrets)
print('MET_OFFICE_GEOHASH (partial):', (MET_OFFICE_GEOHASH[:6] + '...') if MET_OFFICE_GEOHASH else None)
print('MET_OFFICE_LAND_API_BASE:', MET_OFFICE_LAND_API_BASE)

MET_OFFICE_GEOHASH (partial): gcj8ds...
MET_OFFICE_LAND_API_BASE: https://data.hub.api.metoffice.gov.uk


In [7]:
# --- Fetch: get_land_observations(geohash) ---
import requests as _requests


def get_land_observations(geohash, params=None, base=None):
    params = params or {}
    base = base or MET_OFFICE_LAND_API_BASE
    url = f"{base.rstrip('/')}/observation-land/1/{geohash}"
    headers = {'accept': 'application/json', 'apikey': MET_OFFICE_API_KEY}
    r = _requests.get(url, params=params, headers=headers, timeout=20)
    if r.status_code >= 400:
        # include body for diagnostics
        print('Request failed:', r.status_code, r.reason)
        print('Response body (first 1000 chars):')
        print(r.text[:1000])
        r.raise_for_status()
    return r.json()

# run fetch (example)
geohash = MET_OFFICE_GEOHASH
data = get_land_observations(geohash, params={'res': 'hourly'})
print('Fetched top-level type:', type(data))

Fetched top-level type: <class 'list'>


In [8]:
# --- Parse: convert `data` to a pandas DataFrame ---
import json
import pandas as pd

df = None
if 'data' in globals() and data is not None:
    # If the API returned a list of records, normalize directly
    if isinstance(data, list):
        try:
            df = pd.json_normalize(data)
        except Exception as e:
            print('Failed to json_normalize list:', e)
    elif isinstance(data, dict):
        # try to find a list-of-dicts in common keys
        def find_list_of_dicts(d):
            if isinstance(d, list) and d and isinstance(d[0], dict):
                return d
            if isinstance(d, dict):
                for key in ('observations', 'features', 'items', 'data', 'SiteRep'):
                    v = d.get(key)
                    if v is not None:
                        res = find_list_of_dicts(v)
                        if res:
                            return res
                for v in d.values():
                    res = find_list_of_dicts(v)
                    if res:
                        return res
            return None
        lod = find_list_of_dicts(data)
        if lod:
            try:
                df = pd.json_normalize(lod)
            except Exception as e:
                print('Failed to json_normalize found list:', e)

if df is None:
    print('No tabular data found. Sample of `data`:')
    try:
        print(json.dumps(data, indent=2)[:2000])
    except Exception:
        print(repr(data)[:2000])
else:
    display(df.head())

,weather_code,wind_direction,temperature,wind_gust,visibility,mslp,pressure_tendency,datetime,wind_speed,humidity
0,8,SSW,12.04,12.4,12560.0,1015,F,2026-04-04T12:00:00Z,7.7,74
1,8,SW,12.43,12.4,12080.0,1014,F,2026-04-04T13:00:00Z,8.5,78
2,11,SSW,11.40,13.4,3470.0,1013,F,2026-04-04T14:00:00Z,8.1,91
3,12,SSW,12.19,13.9,12000.0,1011,F,2026-04-04T15:00:00Z,8.9,85
4,8,SSW,12.18,18.3,15660.0,1010,F,2026-04-04T16:00:00Z,10.0,81
